# FS-004: MedGemma-4B Baseline on Kaggle (T4 GPU)

This notebook evaluates Google MedGemma-4B (`google/medgemma-4b-it`) on 10 genuine English samples
from the official SLAKE test split, enabling a controlled comparison against the Qwen2.5-VL-3B baseline.

## Key constraints
- **No synthetic images** — real SLAKE dataset images only.
- **No paid APIs or services** — runs locally on Kaggle T4 GPU via Hugging Face Transformers + bitsandbytes.
- **No hardcoded tokens** — `HF_TOKEN` is read from Kaggle Secrets only.
- **No silent CPU/mock fallback** — GPU OOM and other errors are reported clearly.
- **Preserves Qwen baseline** — does not modify existing artifacts or sample manifest.

## Kaggle setup
1. Accelerator: T4 GPU (single GPU, ~14.56 GB VRAM)
2. Add secret: `HF_TOKEN` = your Hugging Face token (Kaggle → Secrets)
3. Accept the MedGemma-4B-IT license on Hugging Face Hub: https://huggingface.co/google/medgemma-4b-it
4. Upload the SLAKE dataset and project code (see Step 2)

## Outputs
Saved under `/kaggle/working/medgemma_baseline/` — export after the session completes.

## Step 1: Verify Kaggle GPU, CUDA, and GPU memory

In [1]:
import torch
import os
import sys
import shutil

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available! Kaggle session must use T4 GPU accelerator. "
        "Set: Settings → Accelerator → T4 x1."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"GPU memory total: {gpu_total:.2f} GB")

# Fallback to local directory if Kaggle path does not exist
disk_path = "/content" if os.path.exists("/content") else "."
total, used, free = shutil.disk_usage(disk_path)
print(f"Disk path checked: {os.path.abspath(disk_path)}")
print(f"Disk total: {total / 1024**3:.1f} GB")
print(f"Disk used: {used / 1024**3:.1f} GB")
print(f"Disk free: {free / 1024**3:.1f} GB")

if gpu_total < 10:
    print("WARNING: Less than 10 GB GPU VRAM. 4-bit MedGemma-4B may OOM.")
if free < 20 * 1024**3:
    print("WARNING: Less than 20 GB free disk. Model download may fail.")

print("\nGPU verification passed.")

PyTorch: 2.14.1+cu130
CUDA available: True
CUDA version: 13.0
GPU: Tesla T4
GPU memory total: 14.56 GB
Disk path checked: /content
Disk total: 112.6 GB
Disk used: 0.2 GB
Disk free: 112.4 GB

GPU verification passed.


## Step 2: Install only required dependencies

Installs `transformers`, `accelerate`, `bitsandbytes`, `safetensors`, `huggingface_hub`, `pillow`, and `numpy`.
Does **not** upgrade or replace Kaggle's existing PyTorch/CUDA installation.

In [2]:
!pip install -q -U transformers accelerate bitsandbytes safetensors huggingface_hub pillow numpy requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


## Step 3: Set up project code and SLAKE dataset

Two options for getting project code onto Kaggle:

**Option A — Upload project archive (recommended):**
1. Zip the MedicalVQA project folder locally (excluding `data/`, `.venv/`, `.git/`)
2. Upload the zip to Kaggle → Add data
3. Extract to `/kaggle/working/MedicalVQA/`

**Option B — Upload individual files:**
Upload `vlm/`, `core/`, `datasets/`, `pyproject.toml`, and the SLAKE dataset.

The SLAKE dataset must be at `/kaggle/working/slake/` with `imgs/` and `test.json`.
Do **not** modify the original SLAKE files.

In [10]:
from pathlib import Path
import zipfile

slake_zip = Path("/content/Slake.zip")
extract_dir = Path("/content/Slake")

# Extract ZIP contents
if slake_zip.exists():
    with zipfile.ZipFile(slake_zip, "r") as archive:

        # Check integrity before extraction
        if archive.testzip() is not None:
            raise RuntimeError("ZIP archive is corrupted.")

        print("Extracting SLAKE dataset...")

        archive.extractall(extract_dir)

        print("Extraction completed successfully!")

# Verify extracted contents
print("\nExtraction directory:", extract_dir)
print("Directory exists:", extract_dir.exists())

files = list(extract_dir.rglob("*"))

print("Total entries:", len(files))
print("Total files:", sum(f.is_file() for f in files))
print("Total folders:", sum(f.is_dir() for f in files))

print("\nFirst 20 extracted items:")

for item in files[:20]:
    print(item.relative_to(extract_dir))

Extracting SLAKE dataset...
Extraction completed successfully!

Extraction directory: /content/Slake
Directory exists: True
Total entries: 3259
Total files: 2604
Total folders: 655

First 20 extracted items:
Slake1.0
__MACOSX
Slake1.0/mask.txt
Slake1.0/test.json
Slake1.0/validate.json
Slake1.0/train.json
Slake1.0/KG
Slake1.0/.DS_Store
Slake1.0/imgs
__MACOSX/Slake1.0
__MACOSX/Slake1.0/._.DS_Store
__MACOSX/Slake1.0/._mask.txt
__MACOSX/Slake1.0/imgs
__MACOSX/Slake1.0/imgs/xmlab259
__MACOSX/Slake1.0/imgs/xmlab554
__MACOSX/Slake1.0/imgs/xmlab419
__MACOSX/Slake1.0/imgs/xmlab199
__MACOSX/Slake1.0/imgs/xmlab562
__MACOSX/Slake1.0/imgs/xmlab278
__MACOSX/Slake1.0/imgs/xmlab290


## Step 4: Verify imports (before downloading model weights)

Verifies `vlm.medgemma_client`, `vlm.base`, `datasets.slake_loader` are importable.
No model weights are downloaded in this step.

In [12]:
import os
import sys

# Dynamically search and append the project folder to python path
possible_roots = [
    "/content/MedicalVQA",
    "/content",
    ".",
    "/kaggle/working/MedicalVQA"
]

project_added = False
for path in possible_roots:
    if os.path.exists(os.path.join(path, "vlm")) and os.path.exists(os.path.join(path, "datasets")):
        abs_path = os.path.abspath(path)
        if abs_path not in sys.path:
            sys.path.insert(0, abs_path)
        os.chdir(abs_path)
        print(f"Found and configured project root at: {abs_path}")
        project_added = True
        break

if not project_added:
    print("WARNING: Could not automatically locate the 'vlm' or 'datasets' folders. "
          "Please make sure you have uploaded the MedicalVQA folder or individual directories to this environment.")

from vlm.medgemma_client import MedGemmaVLMClient
from vlm.base import VLMClient
from datasets.slake_loader import SlakeDatasetLoader

print("All imports successful!")
print(f"MedGemmaVLMClient: {MedGemmaVLMClient}")
print(f"VLMClient: {VLMClient}")
print(f"SlakeDatasetLoader: {SlakeDatasetLoader}")

# Verify MedGemmaVLMClient implements the full VLMClient interface
client_probe = MedGemmaVLMClient()
print(f"\nmodel_name: {client_probe.model_name}")
print(f"device: {client_probe.device}")
print(f"Is VLMClient subclass: {isinstance(client_probe, VLMClient)}")
print(f"generate method: {hasattr(client_probe, 'generate')}")
print(f"sample_n method: {hasattr(client_probe, 'sample_n')}")
print(f"generate_structured method: {hasattr(client_probe, 'generate_structured')}")

Found and configured project root at: /content/MedicalVQA
All imports successful!
MedGemmaVLMClient: <class 'vlm.medgemma_client.MedGemmaVLMClient'>
VLMClient: <class 'vlm.base.VLMClient'>
SlakeDatasetLoader: <class 'datasets.slake_loader.SlakeDatasetLoader'>

model_name: medgemma-4b
device: cpu
Is VLMClient subclass: True
generate method: True
sample_n method: True
generate_structured method: True


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 5: Hugging Face authentication (Kaggle Secrets)

Reads `HF_TOKEN` from the environment (set via Kaggle Secrets).
**Never hardcodes, prints, or saves the token.**

Before running, ensure:
1. Kaggle Secret `HF_TOKEN` is configured with your Hugging Face token
2. You have accepted the MedGemma-4B-IT license: https://huggingface.co/google/medgemma-4b-it

In [14]:

import os
from huggingface_hub import login

# 1. Retrieve Hugging Face token from Colab Secrets
hf_token = None

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")

    if hf_token:
        print("HF_TOKEN found in Google Colab Secrets.")

except Exception:
    print("Colab Secrets unavailable. Checking environment variables...")

# 2. Fallback to environment variables
if not hf_token:
    hf_token = (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    )

# 3. Authenticate with Hugging Face
if hf_token:
    login(
        token=hf_token,
        add_to_git_credential=False
    )
    print("Hugging Face authentication successful.")

else:
    raise RuntimeError(
        "HF_TOKEN not found.\n"
        "To configure it in Google Colab:\n"
        "1. Open the Secrets tab (key icon) in the left sidebar.\n"
        "2. Add a secret named HF_TOKEN.\n"
        "3. Paste your Hugging Face access token.\n"
        "4. Enable notebook access for this secret.\n"
        "5. Run this cell again."
    )


HF_TOKEN found in Google Colab Secrets.
Hugging Face authentication successful.


## Step 6: Verify model repository access and estimate size

Checks that `google/medgemma-4b-it` is accessible and estimates download size.
Does not download model weights yet.

In [15]:
from huggingface_hub import model_info

MODEL_ID = "google/medgemma-4b-it"

try:
    info = model_info(MODEL_ID, files_metadata=True)
    print(f"Model ID: {info.id}")
    print(f"Model sha: {info.sha}")
    print(f"Last modified: {info.lastModified}")
    print(f"Gated: {info.gated}")

    total_size = sum(s.size for s in info.siblings if s.size)
    print(f"Estimated download size: {total_size / 1024**3:.2f} GB")

    if total_size > 30 * 1024**3:
        print("WARNING: Model download exceeds 30 GB. May not fit on Kaggle disk.")
    else:
        print("Model size estimate is acceptable for Kaggle working directory.")
except Exception as e:
    raise RuntimeError(
        f"Could not access model repository '{MODEL_ID}': {type(e).__name__}: {e}. "
        "Ensure you have accepted the MedGemma-4B-IT license on Hugging Face Hub "
        "and configured HF_TOKEN in Kaggle Secrets."
    )

Model ID: google/medgemma-4b-it
Model sha: 290cda5eeccbee130f987c4ad74a59ae6f196408
Last modified: 2025-10-28 17:24:49+00:00
Gated: auto
Estimated download size: 8.05 GB
Model size estimate is acceptable for Kaggle working directory.


In [16]:
import gc, time
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

# NOTE: must be the model you were granted access to (NOT medgemma-1.5-4b-it)
MODEL_ID = "google/medgemma-4b-it"

SYSTEM_PROMPT = "You are an expert radiologist."

# Short-answer JSON instruction so exact-match scoring (extract_answer) works.
ANSWER_INSTRUCTION = (
    "Answer the question about this medical image. "
    "Respond with ONLY a JSON object of the form {\"answer\": \"<answer>\"}. "
    "For yes/no questions the answer must be \"Yes\" or \"No\". "
    "Otherwise give a single word or very short phrase."
)


class MedGemmaClient:
    """MedGemma loaded exactly as in the Hugging Face model card.

    Why this fixes the empty outputs:
      * Gemma 3 / MedGemma is numerically unstable in float16 (activations overflow
        -> NaN -> only pad tokens -> decoded to ''). The T4 has no native bf16, but
        bf16 still works (emulated) and is what the model was trained/released in.
      * attn_implementation="eager" is the recommended setting for Gemma 3.
      * Only the NEWLY generated tokens are decoded (prompt tokens are sliced off).
    """

    def __init__(self, model_id=MODEL_ID, quantization=None, device_index=0):
        self.model_id = model_id
        self.quantization = quantization      # None (bf16, ~8.6 GB) or "4bit" (~3 GB)
        self.device_index = device_index
        self.model = None
        self.processor = None

    @property
    def device(self):
        return str(self.model.device) if self.model is not None else "not loaded"

    def load(self):
        kwargs = dict(
            dtype=torch.bfloat16,
            device_map={"": self.device_index},
            attn_implementation="eager",
        )
        if self.quantization == "4bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16,   # NOT float16
            )
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_id, **kwargs).eval()
        self.processor = AutoProcessor.from_pretrained(self.model_id)
        return self

    def generate(self, marked_image, question, temperature=0.0, max_tokens=512, json_answer=True):
        if self.model is None:
            raise RuntimeError("Call load() before generate().")

        user_text = f"{ANSWER_INSTRUCTION}\n\nQuestion: {question}" if json_answer else question
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": marked_image},
                {"type": "text", "text": user_text},
            ]},
        ]

        inputs = self.processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to(self.model.device, dtype=torch.bfloat16)   # casts pixel_values only; ids stay int
        input_len = inputs["input_ids"].shape[-1]

        gen_kwargs = dict(max_new_tokens=max_tokens, do_sample=temperature > 0)
        if temperature > 0:
            gen_kwargs["temperature"] = temperature

        with torch.inference_mode():
            out = self.model.generate(**inputs, **gen_kwargs)

        new_tokens = out[0][input_len:]
        text = self.processor.decode(new_tokens, skip_special_tokens=True).strip()

        if not text:   # fail loudly instead of silently returning ''
            raise RuntimeError(
                f"MedGemma returned an empty response ({new_tokens.numel()} new tokens, "
                f"unique ids: {new_tokens.unique().tolist()[:10]}). "
                "All-pad output usually means fp16 overflow / NaNs."
            )
        return text


print("MedGemmaClient defined. MODEL_ID =", MODEL_ID)

MedGemmaClient defined. MODEL_ID = google/medgemma-4b-it


## Step 7: Load MedGemma-4B on Tesla T4 GPU (4-bit quantization)

Loads `google/medgemma-4b-it` using the existing `MedGemmaVLMClient` adapter.
Configured for single T4 GPU with 4-bit quantization (`bitsandbytes` NF4, double quant)
to fit within ~14.56 GB VRAM.

**Does not silently fall back to CPU or mock inference.** If loading fails, the notebook stops.

In [17]:
import time, gc
import torch

USE_4BIT = False   # True -> NF4 4-bit with bf16 compute
QUANT_LABEL = "4-bit (NF4, double quant, bf16 compute)" if USE_4BIT else "none (bfloat16)"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. Select a T4 GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Free anything left over from earlier attempts
for _name in ("client", "pipe"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

client = MedGemmaClient(model_id=MODEL_ID, quantization="4bit" if USE_4BIT else None)

t0 = time.time()
try:
    client.load()
    print(f"\nModel loaded successfully in {time.time() - t0:.1f}s")
    print(f"Device: {client.device}")
    print(f"Quantization: {QUANT_LABEL}")
except torch.cuda.OutOfMemoryError as oom_err:
    torch.cuda.empty_cache()
    raise RuntimeError(
        f"GPU out-of-memory while loading MedGemma-4B: {oom_err}. "
        "Restart the runtime, or set USE_4BIT = True. Do not fall back to CPU."
    ) from oom_err
except Exception as exc:
    raise RuntimeError(f"Failed to load MedGemma-4B: {type(exc).__name__}: {exc}") from exc

allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("\n--- GPU Memory Report ---")
print(f"GPU allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | total: {total:.2f} GB")
print("WARNING: GPU memory nearly full." if reserved > total * 0.9 else "GPU memory status: OK")

GPU: Tesla T4
GPU Memory: 14.56 GB


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]


Model loaded successfully in 151.2s
Device: cuda:0
Quantization: none (bfloat16)

--- GPU Memory Report ---
GPU allocated: 8.01 GB | reserved: 8.01 GB | total: 14.56 GB
GPU memory status: OK


## Step 8: One-sample smoke inference (verify before full run)

Runs inference on a single genuine SLAKE image to verify the model processes
image input correctly before proceeding to the full 10-sample baseline.

In [21]:
from PIL import Image
import json

# Use the first sample from the Qwen baseline manifest
MANIFEST_PATH = os.path.join("/content/MedicalVQA", "experiments", "runs", "slake_baseline_ollama", "selection_manifest.json")

with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

smoke_sample = manifest["samples"][0]
print(f"Smoke sample: qid={smoke_sample['qid']}, img={smoke_sample['img_name']}")
print(f"Question: {smoke_sample['question']}")
print(f"Ground truth: {smoke_sample['answer']}")

smoke_image_path = os.path.join("/content/Slake/Slake1.0/", "imgs", smoke_sample["img_name"])
if not os.path.exists(smoke_image_path):
    raise FileNotFoundError(f"Smoke test image not found: {smoke_image_path}")

smoke_image = Image.open(smoke_image_path).convert("RGB")
print(f"Image size: {smoke_image.size}, mode: {smoke_image.mode}")

t0 = time.time()
try:
    smoke_response = client.generate(
        marked_image=smoke_image,
        question=smoke_sample["question"],
        temperature=0.0,
        max_tokens=512,
    )
    smoke_elapsed = time.time() - t0
    print(f"\nRaw response:\n{smoke_response}")
    print(f"\nSmoke inference time: {smoke_elapsed:.4f}s")
    print("\nSmoke test PASSED — model processed the image and generated a response.")
except torch.cuda.OutOfMemoryError as oom_err:
    raise RuntimeError(
        f"GPU out-of-memory during smoke inference: {oom_err}. "
        "Do not proceed to full baseline."
    ) from oom_err
except Exception as exc:
    raise RuntimeError(
        f"Smoke inference failed: {type(exc).__name__}: {exc}. "
        "Do not proceed to full baseline."
    ) from exc

Smoke sample: qid=11981, img=xmlab130/source.jpg
Question: Which part of the body does this image belong to?
Ground truth: Chest
Image size: (1024, 1024), mode: RGB

Raw response:
```json
{"answer": "Chest"}
```

Smoke inference time: 5.2174s

Smoke test PASSED — model processed the image and generated a response.


## Step 9: Load the exact 10 SLAKE samples from the Qwen baseline manifest

Uses the same selection manifest as the Qwen baseline:
`experiments/runs/slake_baseline_ollama/selection_manifest.json`

10 samples (5 OPEN + 5 CLOSED), 9 unique images, seed=42.
Smoke-test samples (qids 11934, 11938, 11965) are excluded per the manifest.

In [22]:
import os
BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
OUTPUT_DIR = os.path.join(BASE_DIR, "medgemma_baseline")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output dir:", OUTPUT_DIR)

samples = manifest["samples"]
print(f"Selected {len(samples)} samples from Qwen baseline manifest")
print(f"Seed: {manifest['seed']}")
print(f"Split: {manifest['split']}, Language: {manifest['language']}")
print(f"Excluded qids: {manifest['excluded_qids']}")
print(f"Unique images: {manifest['unique_images']}")
print()
for i, s in enumerate(samples):
    print(f"  {i+1}. qid={s['qid']} img_id={s['img_id']} img={s['img_name']}")
    print(f"      modality={s['modality']} type={s['answer_type']}")
    print(f"      Q: {s['question']}")
    print(f"      GT: {s['answer']}")

Output dir: /content/medgemma_baseline
Selected 10 samples from Qwen baseline manifest
Seed: 42
Split: test, Language: en
Excluded qids: [11934, 11938, 11965]
Unique images: [130, 185, 224, 234, 312, 320, 344, 561, 605]

  1. qid=11981 img_id=130 img=xmlab130/source.jpg
      modality=X-Ray type=OPEN
      Q: Which part of the body does this image belong to?
      GT: Chest
  2. qid=12073 img_id=185 img=xmlab185/source.jpg
      modality=X-Ray type=CLOSED
      Q: Are there abnormalities in this image?
      GT: Yes
  3. qid=12111 img_id=224 img=xmlab224/source.jpg
      modality=CT type=CLOSED
      Q: Does the picture contain spleen?
      GT: Yes
  4. qid=12129 img_id=234 img=xmlab234/source.jpg
      modality=CT type=OPEN
      Q: How many kidneys in this image?
      GT: 0
  5. qid=12312 img_id=312 img=xmlab312/source.jpg
      modality=X-Ray type=OPEN
      Q: Where is the abnormality in this image?
      GT: Lower Left Lung
  6. qid=12344 img_id=320 img=xmlab320/source.jpg
     

## Step 10: Run MedGemma-4B inference on all 10 samples

For each sample:
- Loads the real SLAKE image (not synthetic)
- Submits image + question to MedGemma-4B
- Records raw response, parsed answer, ground truth, inference time
- Handles OOM and other errors clearly (no silent fallback)

Answer normalization (same as Qwen baseline): strip whitespace, remove code fences,
collapse whitespace, lowercase, remove trailing punctuation. Original answers and
predictions are preserved verbatim.

In [24]:
import re

def normalize_answer(text: str) -> str:
    """Normalize answer for exact-match comparison (same as Qwen baseline)."""
    if text is None:
        return ""
    t = text.strip()
    if t.startswith("```"):
        lines = [l for l in t.split("\n") if not l.strip().startswith("```")]
        t = "\n".join(lines).strip()
    t = t.strip('"').strip("'").strip()
    t = " ".join(t.split())
    t = t.lower()
    while t and t[-1] in ".,!?;:":
        t = t[:-1]
    return t.strip()

def extract_answer(raw: str) -> str:
    """Extract 'answer' field from model JSON response."""
    if raw is None:
        return ""
    t = raw.strip()
    start = t.find("{")
    end = t.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = t[start:end + 1]
        try:
            import json as _json
            obj = _json.loads(candidate)
            if isinstance(obj, dict) and "answer" in obj:
                return str(obj["answer"])
        except _json.JSONDecodeError:
            pass
    return t

results = []

for i, entry in enumerate(samples):
    qid = entry["qid"]
    img_name = entry["img_name"]
    question = entry["question"]
    gt_answer = entry["answer"]

    print(f"\n{'='*60}")
    print(f"Sample {i+1}/{len(samples)}: qid={qid}, img_id={entry['img_id']}")
    print(f"Image: {img_name} ({entry['modality']}, {entry['answer_type']})")
    print(f"Q: {question}")
    print(f"GT: {gt_answer}")
    print(f"{'='*60}")

    image_path = os.path.join("/content/Slake/Slake1.0/", "imgs", img_name)

    if not os.path.exists(image_path):
        print(f"SKIP: Image not found at {image_path}")
        results.append({
            "qid": qid,
            "img_id": entry["img_id"],
            "img_name": img_name,
            "modality": entry["modality"],
            "answer_type": entry["answer_type"],
            "question": question,
            "ground_truth_answer": gt_answer,
            "raw_response": None,
            "parsed_answer": None,
            "normalized_prediction": None,
            "normalized_ground_truth": normalize_answer(gt_answer),
            "correct": False,
            "inference_time_s": None,
            "status": "skipped",
            "error": f"Image not found: {image_path}",
        })
        continue

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as exc:
        print(f"ERROR loading image: {exc}")
        results.append({
            "qid": qid,
            "img_id": entry["img_id"],
            "img_name": img_name,
            "modality": entry["modality"],
            "answer_type": entry["answer_type"],
            "question": question,
            "ground_truth_answer": gt_answer,
            "raw_response": None,
            "parsed_answer": None,
            "normalized_prediction": None,
            "normalized_ground_truth": normalize_answer(gt_answer),
            "correct": False,
            "inference_time_s": None,
            "status": "error",
            "error": f"Failed to load image: {exc}",
        })
        continue

    t_inf = time.time()
    error = None
    raw_response = None

    try:
        raw_response = client.generate(
            marked_image=image,
            question=question,
            temperature=0.0,
            max_tokens=512,
        )
        elapsed = time.time() - t_inf
    except torch.cuda.OutOfMemoryError as oom_err:
        elapsed = time.time() - t_inf
        error = f"GPU OOM: {type(oom_err).__name__}: {oom_err}"
        print(f"GPU OOM: {error}")
    except Exception as exc:
        elapsed = time.time() - t_inf
        error = f"{type(exc).__name__}: {exc}"
        print(f"Inference error: {error}")

    parsed = extract_answer(raw_response) if raw_response and not error else None
    normalized_pred = normalize_answer(parsed) if parsed else None
    normalized_gt = normalize_answer(gt_answer)
    correct = (normalized_pred == normalized_gt) if parsed is not None else False

    status = "error" if error else ("success" if correct is not None else "error")

    print(f"Raw response: {raw_response}")
    print(f"Parsed answer: {parsed}")
    print(f"Correct: {correct}")
    print(f"Inference time: {elapsed:.4f}s")
    if error:
        print(f"Error: {error}")

    results.append({
        "qid": qid,
        "img_id": entry["img_id"],
        "img_name": img_name,
        "modality": entry["modality"],
        "answer_type": entry["answer_type"],
        "question": question,
        "ground_truth_answer": gt_answer,
        "raw_response": raw_response,
        "parsed_answer": parsed,
        "normalized_prediction": normalized_pred,
        "normalized_ground_truth": normalized_gt,
        "correct": correct,
        "inference_time_s": round(elapsed, 4),
        "status": status,
        "error": error,
    })

    # Save individual response
    with open(os.path.join(OUTPUT_DIR, f"q{qid}_response.json"), "w", encoding="utf-8") as rf:
        json.dump(results[-1], rf, indent=2, ensure_ascii=False)


Sample 1/10: qid=11981, img_id=130
Image: xmlab130/source.jpg (X-Ray, OPEN)
Q: Which part of the body does this image belong to?
GT: Chest
Raw response: ```json
{"answer": "Chest"}
```
Parsed answer: Chest
Correct: True
Inference time: 5.9233s

Sample 2/10: qid=12073, img_id=185
Image: xmlab185/source.jpg (X-Ray, CLOSED)
Q: Are there abnormalities in this image?
GT: Yes
Raw response: ```json
{"answer": "Yes"}
```
Parsed answer: Yes
Correct: True
Inference time: 5.0325s

Sample 3/10: qid=12111, img_id=224
Image: xmlab224/source.jpg (CT, CLOSED)
Q: Does the picture contain spleen?
GT: Yes
Raw response: ```json
{"answer": "Yes"}
```
Parsed answer: Yes
Correct: True
Inference time: 5.2580s

Sample 4/10: qid=12129, img_id=234
Image: xmlab234/source.jpg (CT, OPEN)
Q: How many kidneys in this image?
GT: 0
Raw response: {"answer": "1"}
Parsed answer: 1
Correct: False
Inference time: 4.8384s

Sample 5/10: qid=12312, img_id=312
Image: xmlab312/source.jpg (X-Ray, OPEN)
Q: Where is the abnormalit

## Step 11: Save summary and report metrics

Calculates overall, OPEN, and CLOSED accuracy, and saves a machine-readable
summary under `/kaggle/working/medgemma_baseline/`.

In [25]:
total = len(results)
failed = sum(1 for r in results if r["error"] is not None)
valid = total - failed
correct = sum(1 for r in results if r["correct"] is True)
incorrect = valid - correct
accuracy = correct / valid if valid > 0 else 0.0

open_results = [r for r in results if r["answer_type"] == "OPEN" and r["error"] is None]
closed_results = [r for r in results if r["answer_type"] == "CLOSED" and r["error"] is None]
open_correct = sum(1 for r in open_results if r["correct"])
closed_correct = sum(1 for r in closed_results if r["correct"])
open_acc = open_correct / len(open_results) if open_results else None
closed_acc = closed_correct / len(closed_results) if closed_results else None

avg_time = sum(r["inference_time_s"] for r in results if r["inference_time_s"] is not None) / max(valid, 1)

summary = {
    "experiment": "FS-004 MedGemma-4B SLAKE baseline on Kaggle (pilot)",
    "model_id": MODEL_ID,
    "device": f"cuda ({gpu_name})",
    "dtype": "bfloat16",
    "attn_implementation": "eager",
    "quantization": QUANT_LABEL,
    "gpu_memory_gb": round(gpu_total, 2),
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "seed": manifest["seed"],
    "split": manifest["split"],
    "language": manifest["language"],
    "num_selected": total,
    "num_open": len(open_results),
    "num_closed": len(closed_results),
    "unique_images": len(set(r["img_id"] for r in results)),
    "correct": correct,
    "incorrect": incorrect,
    "failed": failed,
    "accuracy": round(accuracy, 4),
    "open_accuracy": round(open_acc, 4) if open_acc is not None else None,
    "closed_accuracy": round(closed_acc, 4) if closed_acc is not None else None,
    "avg_inference_time_s": round(avg_time, 4),
    "results": results,
}

summary_path = os.path.join(OUTPUT_DIR, "baseline_summary.json")
with open(summary_path, "w", encoding="utf-8") as sf:
    json.dump(summary, sf, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print("MEDGEMMA-4B BASELINE SUMMARY")
print(f"{'='*60}")
print(f"Total samples: {total}")
print(f"Correct: {correct}, Incorrect: {incorrect}, Failed: {failed}")
print(f"Overall accuracy: {accuracy:.4f} ({correct}/{valid})")
print(f"OPEN accuracy: {open_acc} ({open_correct}/{len(open_results)})")
print(f"CLOSED accuracy: {closed_acc} ({closed_correct}/{len(closed_results)})")
print(f"Avg inference time: {avg_time:.4f}s")
print(f"\nSummary saved to: {summary_path}")
print(f"Individual responses saved to: {OUTPUT_DIR}/")

# Pilot warning
print("\n" + "!" * 60)
print("PILOT WARNING: 10 samples does not establish statistically reliable benchmark performance.")
print("Do not claim clinical accuracy based on this small experiment.")
print("!" * 60)


MEDGEMMA-4B BASELINE SUMMARY
Total samples: 10
Correct: 7, Incorrect: 3, Failed: 0
Overall accuracy: 0.7000 (7/10)
OPEN accuracy: 0.4 (2/5)
CLOSED accuracy: 1.0 (5/5)
Avg inference time: 5.3056s

Summary saved to: /content/medgemma_baseline/baseline_summary.json
Individual responses saved to: /content/medgemma_baseline/

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
PILOT WARNING: 10 samples does not establish statistically reliable benchmark performance.
Do not claim clinical accuracy based on this small experiment.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
